In [1]:
import numpy as np
import pandas as pd

#  NUMPY BASICS: Fast Numerical Operations
#  Array of delivery distances (in km) for 8 shipments
distances_km = np.array([120, 450, 85, 310, 620, 150, 275, 490])

# Vectorized calculation: Convert km to miles and estimate fuel cost (₹14/km) in one step
distances_miles = distances_km * 0.621371
fuel_costs = distances_km * 14.0

print(f"Distances (km): {distances_km}")
print(f"Mean Distance: {np.mean(distances_km):.2f} km")
print(f"Median Distance: {np.median(distances_km):.2f} km")
print(f"Standard Deviation: {np.std(distances_km):.2f} km")
print(f"Max Fuel Cost: ₹{np.max(fuel_costs):.2f}")

Distances (km): [120 450  85 310 620 150 275 490]
Mean Distance: 312.50 km
Median Distance: 292.50 km
Standard Deviation: 180.83 km
Max Fuel Cost: ₹8680.00


In [2]:
# . CREATE & IMPORT CSV DATASET
raw_csv_content = """tracking_id,origin,destination,weight_kg,shipping_cost,delivery_days,status
TRK201,Chennai,Bengaluru,4.5,350,2,Delivered
TRK202,Mumbai,Delhi,15.0,1200,5,Delivered
TRK203,Chennai,Hyderabad,,420,3,Delayed
TRK204,Bengaluru,Kolkata,8.2,,4,In Transit
TRK205,Delhi,Mumbai,12.0,980,,Delivered
TRK202,Mumbai,Delhi,15.0,1200,5,Delivered
TRK206,Hyderabad,chennai,3.0,280,2,delivered
TRK207,Kolkata,Bengaluru,,650,6,Delayed
TRK208,Chennai,Mumbai,22.5,1850,4,Delivered
"""

with open("messy_shipments.csv", "w") as f:
    f.write(raw_csv_content)

# Import csv into panda dataframe
df = pd.read_csv("messy_shipments.csv")
print("✅ Dataset Imported Successfully!")
df.head(10)  

✅ Dataset Imported Successfully!


,tracking_id,origin,destination,weight_kg,shipping_cost,delivery_days,status
0,TRK201,Chennai,Bengaluru,4.5,350.0,2.0,Delivered
1,TRK202,Mumbai,Delhi,15.0,1200.0,5.0,Delivered
2,TRK203,Chennai,Hyderabad,NaN,420.0,3.0,Delayed
3,TRK204,Bengaluru,Kolkata,8.2,NaN,4.0,In Transit
4,TRK205,Delhi,Mumbai,12.0,980.0,NaN,Delivered
5,TRK202,Mumbai,Delhi,15.0,1200.0,5.0,Delivered
6,TRK206,Hyderabad,chennai,3.0,280.0,2.0,delivered
7,TRK207,Kolkata,Bengaluru,NaN,650.0,6.0,Delayed
8,TRK208,Chennai,Mumbai,22.5,1850.0,4.0,Delivered


In [ ]:
#  EXPLORATORY DATA INSPECTION
print("--- Dataset Shape (Rows, Columns) ---")
print(df.shape)

print("\n--- Column Data Types & Non-Null Counts ---")
df.info()

print("\n--- Missing Values Per Column ---")
print(df.isnull().sum())

print("\n--- Statistical Summary ---")
df.describe()

In [3]:
#  DATA CLEANING & ORGANIZATION
# Step A: Remove duplicate rows
df = df.drop_duplicates()

# Step B: Standardize text formatting (Title Case)
df["destination"] = df["destination"].str.strip().str.title()
df["status"] = df["status"].str.strip().str.title()

# Step C: Handle missing values (Imputation with Median)
df["weight_kg"] = df["weight_kg"].fillna(df["weight_kg"].median())
df["shipping_cost"] = df["shipping_cost"].fillna(df["shipping_cost"].median())
df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

# Step D: Feature Engineering - Add a 'cost_per_kg' column
df["cost_per_kg"] = (df["shipping_cost"] / df["weight_kg"]).round(2)

# Verify zero missing values remain
print("Missing values after cleaning:\n", df.isnull().sum())
df

Missing values after cleaning:
 tracking_id      0
origin           0
destination      0
weight_kg        0
shipping_cost    0
delivery_days    0
status           0
cost_per_kg      0
dtype: int64


,tracking_id,origin,destination,weight_kg,shipping_cost,delivery_days,status,cost_per_kg
0,TRK201,Chennai,Bengaluru,4.5,350.0,2.0,Delivered,77.78
1,TRK202,Mumbai,Delhi,15.0,1200.0,5.0,Delivered,80.00
2,TRK203,Chennai,Hyderabad,10.1,420.0,3.0,Delayed,41.58
3,TRK204,Bengaluru,Kolkata,8.2,650.0,4.0,In Transit,79.27
4,TRK205,Delhi,Mumbai,12.0,980.0,4.0,Delivered,81.67
6,TRK206,Hyderabad,Chennai,3.0,280.0,2.0,Delivered,93.33
7,TRK207,Kolkata,Bengaluru,10.1,650.0,6.0,Delayed,64.36
8,TRK208,Chennai,Mumbai,22.5,1850.0,4.0,Delivered,82.22


In [4]:
#  DATA ANALYSIS & EXPORT
# Analysis 1: Average shipping cost and delivery time by Status
status_summary = df.groupby("status")[["weight_kg", "shipping_cost", "delivery_days"]].mean().round(2)
print("--- Average Metrics by Delivery Status ---")
print(status_summary)

# Analysis : Total revenue generated per Origin city
origin_revenue = df.groupby("origin")["shipping_cost"].sum().sort_values(ascending=False)
print("\n--- Total Shipping Revenue by Origin City (INR) ---")
print(origin_revenue)

# Analysis : Filter high-value shipments (> ₹1000)
high_value = df[df["shipping_cost"] > 1000]
print(f"\nHigh-Value Shipments Count: {len(high_value)}")

# Export the cleaned dataset to a new CSV
df.to_csv("cleaned_shipments.csv", index=False)
print("\n✅ Saved cleaned data to 'cleaned_shipments.csv'!")

--- Average Metrics by Delivery Status ---
            weight_kg  shipping_cost  delivery_days
status                                             
Delayed          10.1          535.0            4.5
Delivered        11.4          932.0            3.4
In Transit        8.2          650.0            4.0

--- Total Shipping Revenue by Origin City (INR) ---
origin
Chennai      2620.0
Mumbai       1200.0
Delhi         980.0
Bengaluru     650.0
Kolkata       650.0
Hyderabad     280.0
Name: shipping_cost, dtype: float64

High-Value Shipments Count: 2

✅ Saved cleaned data to 'cleaned_shipments.csv'!


In [ ]:
## 📊 Task 2: Data Manipulation & Cleaning (NumPy & Pandas)
* **NumPy:** Performed vectorized array operations and descriptive statistics (mean, median, standard deviation) on shipment distances and fuel costs.
* **Data Exploration:** Inspected dataset dimensions, data types, and missing value distributions using `.info()`, `.describe()`, and `.isnull().sum()`.
* **Data Cleaning:** Removed duplicate records, standardized text casing across categorical columns, and imputed missing numerical values using column medians.
* **Data Analysis:** Engineered a `cost_per_kg` metric and used `.groupby()` aggregations to analyze delivery performance by status and revenue by origin hub.
